# Lesson 12 Lab — AWQ: Protecting Salient Weights in W4A16

**Puzzle:** Can activation statistics tell us which weight channels deserve more protection?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

AWQ begins from the observation that a small subset of weights can dominate model behavior when paired with large activation channels. Rather than minimizing average weight error, it uses activation statistics to search a per-channel scaling that protects salient weights while retaining a hardware-friendly weight-only layout.


## 0. Predict before running

1. Predict whether the largest weight magnitudes alone identify the best channels to protect.
2. Explain why AWQ evaluates layer outputs on held-out activations instead of only weight reconstruction.
3. Predict the shape of error as scaling strength increases from zero to one.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

AWQ studies which weight channels are salient under observed activations and protects them within a weight-only W4A16 deployment path.

- AWQ identifies salient weights through activation-aware evidence.
- Equivalent scaling can move quantization difficulty while leaving the original floating-point function unchanged.
- W4A16 describes weight and activation precision; it does not mean the full graph is four-bit.


## 2. Derive the mechanism

Channel scaling can preserve the floating-point linear transform while changing how weight ranges are shared before INT4 rounding. Activation statistics guide the scale search because frequently excited channels can amplify small weight errors.

For a linear layer, equivalent channel scaling can transform weights and inverse-transform activations without changing the floating-point result. AWQ searches a scaling strength informed by activation magnitudes so quantization gives more effective resolution to salient channels. The W4A16 label means four-bit weight storage with floating-point activations; accumulation and other layers still need explicit dtypes.

Scaling too little leaves salient weights exposed. Scaling too aggressively expands other channels and makes their shared quantization ranges coarse. The optimum is therefore empirical and depends on calibration coverage, group size, layer distribution, and the held-out objective.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "12-awq"
device = require_cuda()
torch.manual_seed(2026 + 12)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | uniform W4A16 reference quantization at alpha 0 |
| Candidate | activation-aware channel scaling across alpha 0.25–1.0 |
| Held constant | same weights, calibration/held-out split, group quantizer, activation distribution |
| Measurements | held-out layer-output RMSE, MAE, cosine and selected alpha |
| Evidence | `numerical-model` |

**Experiment:** Search activation-aware per-channel scaling strengths for a toy W4A16 layer and compare output error with naive INT4.


## 5. Read the experiment code

The notebook freezes calibration activations, searches scaling strength, and chooses by held-out layer-output error rather than weight error.

The notebook freezes a calibration activation tensor, derives channel importance, searches five scaling strengths, and evaluates every candidate on held-out activations. The best alpha is chosen from output error, not weight error.

The code is an AWQ-inspired numerical model. It does not implement the paper's complete search, protect exactly the same salient set, reorder or pack weights, or dispatch an AWQ CUDA kernel. Those omissions are stated so the mechanism lesson is not confused with backend reproduction.

Only after these variables match the protocol should the cell be executed.


In [2]:
cal=torch.randn(1024,256,device=device); cal[:,::29]*=7; test=torch.randn(512,256,device=device); test[:,::29]*=7
w=torch.randn(192,256,device=device); ref=test@w.t(); rows=[]
for alpha in (0.0,0.25,0.5,0.75,1.0):
    importance=cal.abs().mean(0).clamp_min(1e-5).pow(alpha); scaled=w*importance
    _,_,dq=symmetric_quantize(scaled,bits=4,group_size=64); restored=dq/importance
    rows.append({"alpha":alpha,"heldout_error":error_metrics(ref,test@restored.t())})
best=min(rows,key=lambda r:r["heldout_error"]["rmse"])
result=base_result(12,"numerical-model"); result.update({"alpha_sweep":rows,"best_alpha":best["alpha"],
    "conclusion":"Activation-aware scaling changed held-out W4A16 output error; no production AWQ kernel was claimed."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Selected alpha | 0.250000 |
| Alpha 0 RMSE | 2.771756 |
| Alpha 0.25 RMSE | 2.273520 |
| Alpha 0.5 RMSE | 2.562305 |
| Alpha 1 RMSE | 5.898383 |


## 7. Interpret rather than merely print

Held-out RMSE improved from 2.771756 at alpha 0 to 2.273520 at alpha 0.25, then worsened to 2.562305, 3.748475, and 5.898383 as alpha increased. Cosine similarity followed the same pattern and peaked at 0.996096 for alpha 0.25.

The non-monotonic curve is the lesson: activation-aware protection can help, but more scaling is not more protection once it transfers too much range pressure elsewhere. The selected value is valid only for this frozen toy distribution.

**Inspection rule:** Use held-out output error and a frozen search set. Weight-only mean error is not the optimization target.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. The CUDA numerical experiment isolates an algorithmic mechanism. It is not the paper's complete implementation and does not establish a production kernel speedup.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "alpha_sweep": [
    {
      "alpha": 0.0,
      "heldout_error": {
        "cosine": 0.99425644,
        "mae": 2.19978571,
        "max_abs": 11.98613548,
        "rmse": 2.77175641
      }
    },
    {
      "alpha": 0.25,
      "heldout_error": {
        "cosine": 0.99609649,
        "mae": 1.80269086,
        "max_abs": 10.36782646,
        "rmse": 2.27352023
      }
    },
    {
      "alpha": 0.5,
      "heldout_error": {
        "cosine": 0.99506319,
        "mae": 2.01121306,
        "max_abs": 12.63252926,
        "rmse": 2.5623045
      }
    },
    {
      "alpha": 0.75,
      "heldout_error": {
        "cosine": 0.98953247,
        "mae": 2.91586828,
        "max_abs": 21.55734253,
        "rmse": 3.74847484
      }
    },
    {
      "alpha": 1.0,
      "heldout_error": {
        "cosine": 0.97454381,
        "mae": 4.58608341,
        "max_abs": 32.47526932,
        "rmse": 5.89838266
      }
    }
  ],
  "best_alpha": 0.25,
  "conclusion": "Activation-aware scaling 

## 9. Make the bounded decision

> Activation-aware protection is a model-quality method; deployment speed still requires a compatible W4A16 kernel.

**Acceptance/rollback:** Separate search/calibration from held-out evaluation, report protected fraction and group size, and prove a W4A16 operator executed before making speed claims.

**Failure analysis:** Using one activation batch for both search and final evaluation can overfit the scale. Reporting W4 storage without the higher-precision activation path misstates memory and compute. And a numerical improvement does not imply latency improvement; the online dequantization and packed GEMM path must exist for the chosen shape.


## 10. Extend the evidence

Repeat the search across several calibration domains and report how stable the selected alpha is. Compare magnitude-only, activation-only, and joint rankings at equal average bit width. Then test an official AWQ checkpoint with operator evidence and batch/sequence sweeps in a serving runtime.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
